# Contractor Registration & Task Assignment Workflow

This notebook automates the process of:
1. Reading contractor registration submissions from Survey123 
2. Checking ArcGIS Online for duplicate accounts before provisioning
3. Creating a new Data Editor / Mobile Worker account if needed
4. Finding unassigned tasks at the contractor's site and assigning them

**How to use:** Run cells top to bottom. Only edit the CONFIG cell.

## 1. Imports & GIS Connection

In [ ]:
from arcgis.gis import GIS
from arcgis.features import FeatureLayer, Feature
import pandas as pd
import random
import string
import copy
from datetime import datetime, timezone, timedelta

# from getpass import getpass <--- Uncomment if you want to use username/password authentication

gis = GIS("home")

# username = input("Enter ArcGIS username: ")
# password = getpass("Enter ArcGIS password: ")
# gis = GIS("https://www.arcgis.com", username, password)

## 2. Configuration

**Only edit this cell.** Update the URLs and field names to match your layers.

In [ ]:
CONFIG = {
    # --- Layer URLs ---
    "registration_url": "https://services.arcgis.com/1234/arcgis/rest/services/Registration/FeatureServer/0",
    "task_layer_url": "https://services.arcgis.com/1234/arcgis/rest/services/Task_Layer/FeatureServer/0",
    "credentials_table_url": "https://services.arcgis.com/1234/arcgis/rest/services/Credentials/FeatureServer/0",
    # --- Registration Form Fields ---
    "reg_field_assignment": "field_assignment",
    "reg_assignment_type": "assignment_type",
    "reg_full_name": "full_name",
    "reg_email": "contact_email_address",
    # --- Customizations ---
    "group_name": "Geology Field Tech Group",
    "username_customization": "_mobile_",
    # --- Task Layer Fields ---
    "task_assignee": "esritask_assignee",
    "task_status": "esritask_status",
    "task_type": "esritask_type",
    "assignment_name": "assignment_name",
    # --- Status Codes ---
    "status_unassigned": "0",
    "status_assigned": "1",
    # --- Timezone ---
    "timezone": "America/Los_Angeles",
    # --- User Provisioning ---
    "data_editor_role_name": "Data Editor",
    "target_user_type": "fieldWorkerUT",  # Data Editor / Mobile Worker license
}

## 3. Helper Functions

### `get_registration()`

Queries the Survey123 layer for submissions made in the past hour (configurable via `lookback_hours`).
Only new submissions are returned. 

| Argument | Type | What you're passing |
|---|---|---|
| `gis` | `GIS` | Authenticated session from Cell 1 |
| `registration_url` | `str` | `CONFIG["registration_url"]` |
| `fields` | `dict` | `CONFIG` — field name mappings |
| `lookback_hours` | `int` | How far back to look, default `1` |

**Returns:** `pd.DataFrame` — one row per new submission, or an empty DataFrame if none.

In [5]:
def get_registration(
    gis, registration_url: str, fields: dict, lookback_hours: int = 1
) -> pd.DataFrame:
    """
    Returns only Survey123 submissions from the past N hours.
    Filtering at query time avoids pulling the full submission history
    on every hourly run.

    Args:
        gis:              Authenticated GIS object
        registration_url: Feature layer URL for the Survey123 form
        fields:           CONFIG dict (provides field name keys)
        lookback_hours:   How many hours back to query (default 1)

    Returns:
        DataFrame with one row per new registrant, or empty if none found
    """
    out_fields = [
        fields["reg_field_assignment"],
        fields["reg_assignment_type"],
        fields["reg_full_name"],
        fields["reg_email"],
    ]

    cutoff_utc = datetime.now(timezone.utc) - timedelta(hours=lookback_hours)
    cutoff_str = cutoff_utc.strftime("%Y-%m-%d %H:%M:%S")

    layer = FeatureLayer(registration_url, gis)
    result = layer.query(
        where=f"CreationDate >= TIMESTAMP '{cutoff_str}'",
        out_fields=out_fields,
    )

    if not result.features:
        print(f"No new submissions in the last {lookback_hours} hour(s).")
        return pd.DataFrame(columns=out_fields)

    df = result.sdf[out_fields].copy()
    print(
        f"Found {len(df)} new registrant(s) submitted in the last {lookback_hours} hour(s)."
    )
    display(df)
    return df

### `get_existing_arcgis_users()`

Fetches every user in the ArcGIS Online org in a single call and builds two lookup structures
used to prevent duplicate account creation.

| Check | What is verified |
|---|---|
| **A — Email in org?** | Is the registrant's email already tied to any account? |
| **B — Username available?** | Is the generated username already taken? |
| **C — Same license type?** | If A matches and the account is already `fieldWorkerUT`, skip — no duplicate of the same type. |

**Returns:** `tuple` of `(existing_usernames: set, email_to_user_type: dict)`

In [ ]:
def get_existing_arcgis_users(gis, max_users: int = 10000) -> tuple:
    """
    Fetches all users in the ArcGIS Online org once and returns two
    lookup structures for duplicate detection.

    Args:
        gis:       Authenticated GIS object
        max_users: Maximum users to retrieve (default 10 000)

    Returns:
        Tuple of:
          existing_usernames  — set of all lowercase usernames in the org
          email_to_user_type  — dict of lowercase email -> userLicenseTypeId
    """
    all_users = gis.users.search(max_users=max_users)

    # Usernames stored lowercase — AGOL is case-insensitive for usernames
    existing_usernames: set = set()
    email_to_user_type: dict = {}

    for u in all_users:
        existing_usernames.add(u.username.lower())
        if getattr(u, "email", None):
            email_lower = u.email.strip().lower()
            u_type = getattr(u, "userLicenseTypeId", None)
            email_to_user_type[email_lower] = u_type if u_type is not None else ""

    print(f"Found {len(existing_usernames)} user(s) currently in ArcGIS Online.")
    return existing_usernames, email_to_user_type

### `generate_username()`

Builds a username from the registrant's name using the pattern `<first initial><customization><last name>`,
e.g. `j_mobile_doe`. If that name is taken, appends an incrementing number (`j_mobile_doe2`, `j_mobile_doe3` …)
until a free slot is found.

**Returns:** `str` — a unique username not present in `existing_usernames`

In [7]:
def generate_username(
    full_name: str, existing_usernames: set, username_customization: str
) -> str:
    """
    Derives a unique username from a full name, avoiding collisions
    against the current org's username set.

    Args:
        full_name:              Registrant's full name, e.g. 'Jane Doe'
        existing_usernames:     Set of lowercase usernames already in the org
        username_customization: String inserted between initial and last name

    Returns:
        A unique username string
    """
    parts = full_name.strip().lower().split()
    first = parts[0]
    last = parts[-1] if len(parts) > 1 else parts[0]
    first_initial = "".join(c for c in first[0] if c.isalnum())
    last_clean = "".join(c for c in last if c.isalnum())
    base = first_initial + username_customization + last_clean  # e.g. "j_mobile_doe"

    candidate = base
    counter = 2
    while candidate in existing_usernames:
        candidate = f"{base}{counter}"
        counter += 1
    return candidate

### `generate_password()`

Generates a random 12-character password that satisfies ArcGIS Online's complexity rules:
at least one uppercase, one lowercase, one digit, and one special character.

**Returns:** `str` — e.g. `"Kx#3mWqZ9!pL"`

In [8]:
def generate_password(length: int = 12) -> str:
    """
    Generates a random password meeting ArcGIS Online complexity rules.
    Guarantees at least one uppercase, lowercase, digit, and special character.

    Args:
        length: Total password length (default 12)

    Returns:
        Password string
    """
    lower = string.ascii_lowercase
    upper = string.ascii_uppercase
    digits = string.digits
    special = "!@#$%^&*"

    # Seed with one guaranteed character of each required type
    pwd = [
        random.choice(upper),
        random.choice(lower),
        random.choice(digits),
        random.choice(special),
    ]
    pool = lower + upper + digits + special
    pwd += random.choices(pool, k=length - 4)
    random.shuffle(pwd)
    return "".join(pwd)

### `get_data_editor_role_id()` and `create_mobile_worker()`

`get_data_editor_role_id()` looks up the role ID by name at runtime using `gis.users.roles.all()`
rather than relying on a hardcoded internal ID (e.g. `"iBBBBBBBBBBBBBBB"`) that could silently
break if the org is reconfigured. The role name comes from `CONFIG["data_editor_role_name"]`.

`create_mobile_worker()` uses both CONFIG values — `data_editor_role_name` and `target_user_type` —
so provisioning behaviour is fully controlled from the CONFIG cell.

**Returns:** `User` object if successful, `None` if creation failed

In [9]:
def get_data_editor_role_id(gis, role_name: str) -> str:
    """
    Looks up a role ID by display name.

    Args:
        gis:       Authenticated GIS object
        role_name: Display name of the role, e.g. CONFIG["data_editor_role_name"]

    Returns:
        Role ID string

    Raises:
        ValueError if the role name is not found in the org
    """
    for role in gis.users.roles.all():
        if role.name == role_name:
            return role.role_id
    raise ValueError(
        f"Role '{role_name}' not found in this org. "
        f"Check available roles with: gis.users.roles.all()"
    )


def create_mobile_worker(gis, full_name: str, email: str, username: str, password: str):
    """
    Creates a new ArcGIS Online account for a mobile worker.
    Role ID and user type are read from CONFIG — nothing is hardcoded.

    Args:
        gis:       Authenticated GIS object
        full_name: Registrant's full name, e.g. 'Jane Doe'
        email:     Registrant's email address
        username:  Pre-generated username
        password:  Pre-generated password

    Returns:
        Created User object, or None if creation failed
    """
    name_parts = full_name.strip().split()
    first_name = name_parts[0]
    last_name = " ".join(name_parts[1:]) if len(name_parts) > 1 else ""

    role_id = get_data_editor_role_id(gis, CONFIG["data_editor_role_name"])

    user = gis.users.create(
        username=username,
        password=password,
        firstname=first_name,
        lastname=last_name,
        email=email,
        role=role_id,
        user_type=CONFIG["target_user_type"],
    )

    if user:
        print(f"Created mobile worker: {user.username}")
    else:
        print(f"Failed to create user: {username}")
    return user

### `add_credentials_to_table()`

Appends one row to the hosted credentials table. Accepts a pre-instantiated `FeatureLayer`
so the layer object is built once outside the loop rather than reconstructed on every call.

**Returns:** `bool` — `True` if the insert succeeded, `False` otherwise

In [10]:
def add_credentials_to_table(credentials_layer, record: dict) -> bool:
    """
    Appends one row to the hosted credentials table.

    Args:
        credentials_layer: Pre-instantiated FeatureLayer / table object
        record:            Dict with keys: username, password, firstname,
                           lastname, email, role, user_type

    Returns:
        True if the insert succeeded, False otherwise
    """
    feature = Feature(
        attributes={
            "username": record["username"],
            "password": record["password"],
            "firstname": record["firstname"],
            "lastname": record["lastname"],
            "email": record["email"],
            "role": record["role"],
            "user_type": record["user_type"],
        }
    )

    result = credentials_layer.edit_features(adds=[feature])
    success = result["addResults"][0]["success"] if result.get("addResults") else False

    if success:
        print(f"Credentials for {record['username']} saved to hosted table.")
    else:
        print(f"Failed to save credentials for {record['username']}: {result}")
    return success

### `add_user_to_group()`

Searches the AGOL org for a group matching `CONFIG["group_name"]` and adds the new user to it.
Group membership controls which maps and apps appear in Field Maps.

**Returns:** `bool` — `True` if added, `False` if the group was not found or an error occurred

In [11]:
def add_user_to_group(gis, user, group_name: str) -> bool:
    """
    Adds a user to a named group in ArcGIS Online.

    Args:
        gis:        Authenticated GIS object
        user:       User object to add
        group_name: Display name of the target group

    Returns:
        True if successful, False otherwise
    """
    try:
        groups = gis.groups.search(f"title:{group_name}")
        if not groups:
            print(f"Group '{group_name}' not found.")
            return False
        groups[0].add_users([user.username])
        print(f"Added {user.username} to group '{group_name}'")
        return True
    except Exception as e:
        print(f"Failed to add user to group: {e}")
        return False

### `get_unassigned_tasks()`

Queries the task layer for features where the assignee is `NULL`, status is `0` (Unassigned),
and `assignment_name` matches the contractor's site. Accepts a pre-instantiated `FeatureLayer`
to avoid reconstructing it on every loop iteration.

**Returns:** `list[Feature]` — unassigned features at the given site

In [12]:
def get_unassigned_tasks(task_layer, site_name: str, fields: dict) -> list:
    """
    Returns unassigned task features at a given site.

    Args:
        task_layer: Pre-instantiated FeatureLayer for the task layer
        site_name:  Site name to filter by, e.g. 'Redlands'
        fields:     CONFIG dict — provides field names and status codes

    Returns:
        List of unassigned Feature objects
    """
    assignee_field = fields["task_assignee"]
    status_field = fields["task_status"]
    unassigned = fields["status_unassigned"]
    assignment = fields["assignment_name"]

    where = (
        f"{assignee_field} IS NULL "
        f"AND {status_field} = '{unassigned}' "
        f"AND {assignment} = '{site_name}'"
    )

    features = task_layer.query(where=where).features
    print(f"Found {len(features)} unassigned task(s) at '{site_name}'.")
    return features

### `register_assignee_in_domain()`

The task layer's assignee field is a coded-value dropdown — only values registered in its
domain can be assigned. This function reads the current domain, adds the new username if it
is not already present, and pushes the updated definition back to the layer.

Uses `copy.deepcopy()` to avoid accidentally mutating the live `layer.properties` object.

**Returns:** `None`

In [13]:
def register_assignee_in_domain(
    task_layer, assignee_field: str, display_name: str, code: str
) -> None:
    """
    Adds a new coded value to the assignee field's domain if not already present.
    Uses a deep copy of layer.properties to avoid mutating live state.

    Args:
        task_layer:     Pre-instantiated FeatureLayer (task layer)
        assignee_field: Name of the assignee field
        display_name:   Human-readable label for the dropdown, e.g. full name
        code:           Stored value, typically the username
    """
    # Deep copy — prevents mutating the live layer.properties object in place
    definition = copy.deepcopy(dict(task_layer.properties))

    for field in definition["fields"]:
        if field["name"] == assignee_field and "domain" in field:
            existing_codes = [cv["code"] for cv in field["domain"]["codedValues"]]
            if code not in existing_codes:
                field["domain"]["codedValues"].append(
                    {"name": display_name, "code": code}
                )
                print(f"Added '{display_name}' ({code}) to assignee domain.")
            else:
                print(f"'{code}' already in assignee domain — skipping.")

    task_layer.manager.update_definition({"fields": definition["fields"]})

### `assign_tasks()`

Sets the assignee field and flips the status from `0` (Unassigned) to `1` (Assigned) on every
feature in the list. All updates are sent in a single batch `edit_features()` call.

**Returns:** `None`

In [14]:
def assign_tasks(task_layer, features: list, assignee_code: str, fields: dict) -> None:
    """
    Batch-updates features: assigns the user and marks them as assigned.

    Args:
        task_layer:    Pre-instantiated FeatureLayer (task layer)
        features:      List of Feature objects to update
        assignee_code: Username to assign (coded value)
        fields:        CONFIG dict — provides field names and status codes
    """
    if not features:
        print("No features to update.")
        return

    assignee_field = fields["task_assignee"]
    status_field = fields["task_status"]
    new_status = fields["status_assigned"]

    for feature in features:
        feature.attributes[assignee_field] = assignee_code
        feature.attributes[status_field] = new_status

    result = task_layer.edit_features(updates=features)

    # sum() over booleans is idiomatic — True == 1, False == 0
    successes = sum(r["success"] for r in result["updateResults"])
    failures = len(result["updateResults"]) - successes

    print(f"Updated {successes} task(s) successfully.")
    if failures:
        print(f"{failures} update(s) failed. Full result:")
        print(result)

## 4. Step 1 — Fetch New Registrants from Survey123

Queries Survey123 for submissions made in the **past hour only**.
The result is stored in `new_registrants` — an empty DataFrame means no one submitted this run.

In [ ]:
# Fetch only submissions from the past hour.
# Adjust lookback_hours if you need a wider window.
new_registrants = get_registration(
    gis, CONFIG["registration_url"], CONFIG, lookback_hours=3
)

## 5. Step 2 — Process New Registrants

For each submission in `new_registrants`, the loop:

1. Guards against a missing email (NaN)
2. Checks the live AGOL org for duplicates (email + license type)
3. Generates unique credentials and creates the account
4. Saves credentials to the hosted table
5. Adds the user to the configured group
6. Finds and assigns unassigned tasks at their site

**Skip conditions:**

| Condition | Outcome |
|---|---|
| Email in org + already `fieldWorkerUT` | Skip — duplicate of same type |
| Email in org + different license type | Skip with warning — review manually |
| Email not in org | Proceed with full onboarding |

Both `FeatureLayer` objects are built once before the loop and reused across all iterations.
Each registrant is wrapped in `try/except` so one failure never aborts the rest.

In [ ]:
# ── Build shared FeatureLayer objects once — not rebuilt per iteration ────────
credentials_layer = FeatureLayer(CONFIG["credentials_table_url"], gis)
task_layer = FeatureLayer(CONFIG["task_layer_url"], gis)

# Single network call to snapshot all org users
existing_usernames, email_to_user_type = get_existing_arcgis_users(gis)

new_count = 0

for _, row in new_registrants.iterrows():

    # ── Skip rows missing an email ────────────────────────────────
    raw_email = row[CONFIG["reg_email"]]
    if pd.isna(raw_email):
        print("Skipping row — missing email value.")
        continue
    email = str(raw_email).strip().lower()
    full_name = str(row[CONFIG["reg_full_name"]])

    # ── Duplicate check: email already in org? ────────────────────────────────
    if email in email_to_user_type:
        existing_type = email_to_user_type[email]
        if existing_type == CONFIG["target_user_type"]:
            print(
                f"Skipping {full_name} ({email}) — already exists as a "
                f"{CONFIG['target_user_type']} (Data Editor / Mobile Worker). "
                f"No duplicate account will be created."
            )
        else:
            print(
                f"WARNING: {full_name} ({email}) already has an account with license "
                f"type '{existing_type}' (not {CONFIG['target_user_type']}). "
                f"Skipping — review manually before provisioning."
            )
        continue

    site = str(row[CONFIG["reg_field_assignment"]])
    username_customization = str(CONFIG["username_customization"])

    print(f"Processing: {full_name} ({email})")

    try:
        # 1. Generate unique credentials
        username = generate_username(
            full_name, existing_usernames, username_customization
        )
        password = generate_password()
        existing_usernames.add(username.lower())  # reserve immediately for this run

        name_parts = full_name.strip().split()
        firstname = name_parts[0]
        lastname = " ".join(name_parts[1:]) if len(name_parts) > 1 else ""

        # 2. Create ArcGIS Online account
        new_user = create_mobile_worker(gis, full_name, email, username, password)
        if not new_user:
            print(f"Skipping remaining steps for {full_name} — user creation failed.")
            continue

        # 3. Save credentials to hosted table
        add_credentials_to_table(
            credentials_layer,
            {
                "username": username,
                "password": password,
                "firstname": firstname,
                "lastname": lastname,
                "email": email,
                "role": CONFIG["data_editor_role_name"],
                "user_type": CONFIG["target_user_type"],
            },
        )

        # 4. Add to approved group
        add_user_to_group(gis, new_user, CONFIG["group_name"])

        # 5. Find unassigned tasks at their site
        unassigned_tasks = get_unassigned_tasks(task_layer, site, CONFIG)

        # 6. Register in the assignee domain
        register_assignee_in_domain(
            task_layer=task_layer,
            assignee_field=CONFIG["task_assignee"],
            display_name=full_name,
            code=username,
        )

        # 7. Assign the tasks
        assign_tasks(
            task_layer=task_layer,
            features=unassigned_tasks,
            assignee_code=username,
            fields=CONFIG,
        )

        new_count += 1

    except Exception as e:
        print(f"ERROR processing {full_name} ({email}): {e}")
        print("Skipping to next registrant.")
        continue

print(f"Polling complete. {new_count} new registrant(s) processed.")